# **Modelo LightGCN**
### Proyecto Hito 2
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

## Índice

>[0- Instalación de librerías](#0--instalación-de-librerías)

>[1- Carga de datos](#1--carga-de-datos)

>[2- Definición del modelo, formateo de datos, y entrenamiento](#2--definición-del-modelo-formateo-de-datos-y-entrenamiento)

>[3- Generación de recomendaciones](#3--generación-de-recomendaciones)

>[4- Métricas](#4--métricas)

>[5- Referencias](#5--referencias)

## 0- Instalación de librerías

In [61]:
# !pip uninstall -y numpy
# !pip install numpy==1.26

In [62]:
# !pip install scikit-surprise --no-build-isolation --no-deps

In [63]:
# pip install pandas

## 1- Carga de datos

In [1]:
import surprise
import numpy as np
import pandas as pd
from collections import defaultdict
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy
import random

Se leen los archivos de datos y se almacenan en un dataframe. El primero, df_original, lee el archivo original del dataset. El segundo, "df_con_ids", lee el archivo del dataset modificado con ids de usuario e ítem agregadas, tal como se hizo en el Hito 1 del proyecto al implementar los modelos referenciales (es el mismo archivo creado en /modelos_ref/modelos_ref.ipynb). El tercero, "df_final", es el mismo contenido de "df_con_ids", pero tomando sólo las columnas de id de usuario, id de ítem, y rating, con estos últimos convertidos a escala del 1 al 5, tal como se hizo en el Hito 1 del proyecto al implementar los modelos referenciales (es el mismo archivo creado en /modelos_ref/modelos_ref.ipynb). Los dataframes son:

In [2]:
df_original = pd.read_csv('video_game_reviews.csv')
df_con_ids = pd.read_csv('video_game_reviews_with_userid.csv')
df_final = pd.read_csv('video_game_reviews_with_userid_clean.csv')

En este diccionario "info_videojuegos" se guarda la id del videojuego junto a su título correspondiente, para luego obtener información de este (como su género, plataforma, etc.):

In [3]:
info_videojuegos = dict(zip(df_con_ids['item_id'], df_con_ids['Game Title']))

El dataframe final que se utilizará para las recomendaciones es:

In [15]:
df_final

,user_id,item_id,rating
0,861,12,3.670051
1,1295,38,3.862944
2,1131,21,2.695431
3,1096,4,3.873096
4,1639,14,3.030457
...,...,...,...
47769,1293,21,4.197970
47770,2485,37,2.431472
47771,2675,3,2.685279
47772,2599,37,2.258883


Se lee el .csv definitivo con los datos modificados (tal como el dataframe "df_final") y se definen los datasets de entrenamiento y testeo:

In [10]:
reader = Reader(line_format='user item rating', sep=',', rating_scale=(1,5), skip_lines=1)
data = Dataset.load_from_file('video_game_reviews_with_userid_clean.csv', reader=reader)

trainset, testset = train_test_split(data, test_size=0.2)

print("Usuarios:", trainset.n_users)
print("Items:", trainset.n_items)
print("Test size:", len(testset))

Usuarios: 3000 Items: 40 Test size: 9555


## 2- Definición del modelo, formateo de datos, y entrenamiento

VIEJO (no funciona):

In [ ]:
import os
import pandas as pd
from recbole.quick_start import run_recbole
from recbole.utils.case_study import full_sort_topk

# === 1. Cargar tu dataset CSV ===
df_lightgcn = pd.read_csv("video_game_reviews_with_userid_clean.csv")

# === 2. Crear estructura que RecBole necesita ===

# # RecBole requiere un campo de tiempo, así que añadimos uno aunque LightGCN no lo use
# df_lightgcn["timestamp"] = pd.RangeIndex(start=1, stop=len(df_lightgcn) + 1)

df_lightgcn.columns = [
    "user_id:token",
    "item_id:token",
    "rating:float"
]

dataset_name = "video_game_reviews_recbole"
dataset_dir = os.path.join(".", dataset_name)  # carpeta mínima en el directorio actual

# Crear la carpeta si no existe
os.makedirs(dataset_dir, exist_ok=True)

# Nombre del archivo .inter
inter_file = os.path.join(dataset_dir, f"{dataset_name}.inter")

# Guardar el CSV en la carpeta del dataset
df_lightgcn.to_csv(inter_file, sep="\t", index=False)

# === 3. Configuración para LightGCN ===
config_dict = {
    "data_path": ".",
    "dataset": "video_game_reviews_recbole",
    "dataset_file": {"inter": "video_game_reviews_recbole/video_game_reviews_recbole.inter"},

    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating",
    "load_col": {"inter": ["user_id", "item_id", "rating"]},

    "field_type": {
    "user_id": "token",
    "item_id": "token",
    "rating": "float"
    },

    # División de datos 80/10/10 (train/valid/test)
    "eval_args": {"split": {"RS": [0.8, 0.1, 0.1]}, "mode": "full"},
    "epochs": 10,                    # puedes aumentar para mejor entrenamiento
    "train_batch_size": 2048,
    "eval_batch_size": 4096,
    "embedding_size": 64,
    "show_progress": True,

    # LightGCN hiperparámetros (opcional ajustar)
    "learning_rate": 0.001,
    "reg_weight": 1e-5,
    "n_layers": 3,
    "topk": [10],

    #Evitar creación de carpeta /log y registro de checkpoints:
    # 'show_progress': True,
    # 'save_model': False,        # ❌ No guardar checkpoints
    # 'save_dataset': False,      # ❌ No guardar dataset cache
    # 'save_reproduce': False,    # ❌ No guardar archivos de reproducción
    # 'logger': None,  
}

# === 4. Entrenar LightGCN ===
config, model, dataset, trainer = run_recbole(
    model="LightGCN",
    dataset='video_game_reviews_recbole',
    config_dict=config_dict
)

print("\n Entrenamiento finalizado correctamente.")

# === 5. Generar recomendaciones ===
topk = 10
topk_result = full_sort_topk(model, dataset, k=topk, device=model.device)

# === 6. Convertir a IDs originales (más legible) ===
if isinstance(topk_result, tuple):
    topk_items, topk_scores = topk_result
else:
    topk_items = topk_result

# Mostrar ejemplo: top 5 usuarios con sus 10 recomendaciones
print("\n🎮 Recomendaciones ejemplo:")
user_ids = list(range(5))
for uid in user_ids:
    tokens = [dataset.id2token(dataset.iid_field, iid) for iid in topk_items[uid]]
    print(f"Usuario {dataset.id2token(dataset.uid_field, uid)} → {tokens}")

NUEVO (funciona):

In [31]:
import os
import pandas as pd
import torch

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.trainer import Trainer
from recbole.utils.case_study import full_sort_topk

# === 1. Cargar CSV y renombrar columnas según RecBole ===
df_lightgcn = pd.read_csv("video_game_reviews_with_userid_clean.csv")
df_lightgcn.columns = [
    "user_id:token",
    "item_id:token",
    "rating:float"
]

# === 2. Guardar dataset en la carpeta mínima para RecBole ===
dataset_name = "video_game_reviews_recbole"
dataset_dir = os.path.join(".", dataset_name)
os.makedirs(dataset_dir, exist_ok=True)
inter_file = os.path.join(dataset_dir, f"{dataset_name}.inter")
df_lightgcn.to_csv(inter_file, sep="\t", index=False)

# === 3. Configuración ===
config_dict = {
    "data_path": ".",
    "dataset": dataset_name,
    "dataset_file": {"inter": f"{dataset_name}/{dataset_name}.inter"},
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating",
    "load_col": {"inter": ["user_id", "item_id", "rating"]},
    "field_type": {
        "user_id": "token",
        "item_id": "token",
        "rating": "float"
    },
    "eval_args": {"split": {"RS": [0.8, 0.1, 0.1]}, "mode": "full"},
    "epochs": 10,
    "train_batch_size": 2048,
    "eval_batch_size": 4096,
    "embedding_size": 64,
    "show_progress": True,
    "learning_rate": 0.001,
    "reg_weight": 1e-5,
    "n_layers": 3,
    "topk": [10],
    "device": "cpu"  # o "cuda" si tienes GPU compatible
}

# === 4. Crear config, dataset y dataloaders ===
config = Config(model='LightGCN', dataset=dataset_name, config_dict=config_dict)
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

# === 5. Crear modelo y trainer ===
model = LightGCN(config, train_data.dataset).to(config['device'])
trainer = Trainer(config, model)

# === 6. Entrenar ===
trainer.fit(train_data, valid_data)
print("\n✅ Entrenamiento finalizado.")

# === 7. Generar recomendaciones Top-K para todos los usuarios ===
# Solo usuarios que existen en test_data
dataset_obj = test_data.dataset
# internal IDs de usuarios en test
uid_series = [uid for uid in range(dataset_obj.user_num) 
              if test_data.uid2history_item[uid] is not None]

topk = 10
topk_result = full_sort_topk(uid_series, model, test_data, k=topk, device=config['device'])

# topk_result es torch.return_types.topk
topk_indices = topk_result.indices  # IDs internos de items
topk_scores = topk_result.values    # Scores

# === 8. Convertir a IDs originales (más legible) ===
print("\n🎮 Recomendaciones ejemplo:")
for uid in range(5):
    # Convertir la fila de indices a lista de enteros
    item_indices = topk_indices[uid].tolist()
    item_tokens = [dataset.id2token(dataset.iid_field, int(iid)) for iid in item_indices]
    user_token = dataset.id2token(dataset.uid_field, uid)
    print(f"Usuario {user_token} → {item_tokens}")



c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the interm


✅ Entrenamiento finalizado.

🎮 Recomendaciones ejemplo:
Usuario [PAD] → ['22', '3', '37', '20', '26', '9', '39', '13', '17', '16']
Usuario 861 → ['7', '12', '1', '18', '29', '33', '24', '16', '10', '39']
Usuario 1295 → ['29', '10', '9', '8', '35', '15', '27', '18', '32', '12']
Usuario 1131 → ['29', '6', '30', '20', '10', '22', '39', '15', '9', '37']
Usuario 1096 → ['17', '21', '9', '31', '40', '22', '13', '30', '26', '3']


Se guardan en un diccionario las recomendaciones con las ids de usuario e ítem originales, convirtiendo las internas de RecBole a las originales del dataset. El diccionario tiene como llaves la id del usuario y su valor es una lista con las 10 id de ítems recomendados: 

In [33]:
# Convertir IDs internos de usuarios a originales
user_original_ids = dataset_obj.id2token('user_id', uid_series)

# Convertir IDs internos de items a originales
topk_items_original = [
    dataset_obj.id2token('item_id', topk_result.indices[i])
    for i in range(len(uid_series))
]

# Crear diccionario final
recommendations_dict = {user_id: items for user_id, items in zip(user_original_ids, topk_items_original)}

# Ejemplo
for user_id, items in list(recommendations_dict.items())[:5]:
    print(f"Usuario {user_id} → {items}")


Usuario 861 → ['22' '3' '37' '20' '26' '9' '39' '13' '17' '16']
Usuario 1295 → ['7' '12' '1' '18' '29' '33' '24' '16' '10' '39']
Usuario 1131 → ['29' '10' '9' '8' '35' '15' '27' '18' '32' '12']
Usuario 1096 → ['29' '6' '30' '20' '10' '22' '39' '15' '9' '37']
Usuario 1639 → ['17' '21' '9' '31' '40' '22' '13' '30' '26' '3']


Usuarios con recomendaciones (en total el dataset tiene 3000):

In [34]:
len(recommendations_dict)

3000

Métricas de resultados en test

In [38]:
test_result = trainer.evaluate(test_data)
# test_result es un OrderedDict que devuelve RecBole
final_metrics = dict(test_result)

print(final_metrics)

25 Oct 21:25    INFO  Loading model structure and parameters from saved\LightGCN-Oct-25-2025_21-09-31.pth


{'recall@10': 0.5262, 'mrr@10': 0.3294, 'ndcg@10': 0.3579, 'hit@10': 0.5717, 'precision@10': 0.0635}


## 3- Generación de recomendaciones

## 4- Métricas

## 5- Referencias

- [1] Documentación de RecBole: https://github.com/RUCAIBox/RecBole?utm_source=chatgpt.com